In [2]:
import requests
import pandas as pd
import time
from tqdm import tqdm
import xml.etree.ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.colab import files

##############################################################
# ROBUST CONNECTION SETUP (Prevents Colab Disconnects)
##############################################################
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

##############################################################
# USER SETTINGS
##############################################################

EMAIL = "simon.anguzu@students.mak.ac.ug"       # Required by NCBI
API_KEY = ""                                    # Optional but recommended

# Updated date range: 2010 to 2026
START_YEAR = "2010"
END_YEAR = "2026"

# Comprehensive search query covering bees, pesticides, and omics/expression terms
SEARCH_QUERY = r'''
("honey bee*" OR "honeybee*" OR "Apis" OR "Apis mellifera" OR "Apis cerana" OR "Apis dorsata" OR "stingless bee*" OR "Meliponini" OR "Melipona" OR "Meliponula" OR "bumblebee*" OR "Bombus" OR "Bombus terrestris" OR "Bombus impatiens" OR "non-Apis" OR "wild bee*" OR "pollinator*" OR "Apoidea")
AND
("pesticide*" OR "agrochemical*" OR "insecticide*" OR "neonicotinoid*" OR "imidacloprid" OR "thiamethoxam" OR "clothianidin" OR "acetamiprid" OR "thiacloprid" OR "flupyradifurone" OR "fungicide*" OR "herbicide*" OR "glyphosate" OR "acaricide*" OR "fipronil" OR "organophosphate*" OR "pyrethroid*" OR "carbamate*" OR "chlorpyrifos" OR "deltamethrin")
AND
("transcriptom*" OR "RNA-Seq" OR "RNA sequencing" OR "gene expression" OR "differentially expressed" OR "DEG" OR "DEGs" OR "microarray*" OR "proteom*" OR "metabolom*" OR "multi-omics" OR "multiomics" OR "ecotoxicogenom*" OR "toxicogenom*" OR "next generation sequencing" OR "NGS")
'''

##############################################################
# SEARCH PUBMED (2010–2026 Range)
##############################################################

base_search = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": SEARCH_QUERY,
    "datetype": "pdat",       # Date type: Publication Date
    "mindate": START_YEAR,    # 2010
    "maxdate": END_YEAR,      # Includes 2026 publications
    "retmode": "json",
    "retmax": 0,              # Get count first
    "email": EMAIL
}

if API_KEY:
    params["api_key"] = API_KEY

print("Querying NCBI to find total record count...")
r = session.get(base_search, params=params)
r.raise_for_status()

data = r.json()
count = int(data["esearchresult"]["count"])

print(f"\nTotal PubMed records found ({START_YEAR}-{END_YEAR}): {count}\n")

##############################################################
# DOWNLOAD ALL PMIDs
##############################################################

pmids = []
batch_size = 500

for start in tqdm(range(0, count, batch_size), desc="Fetching PMIDs"):
    search_params = {
        "db": "pubmed",
        "term": SEARCH_QUERY,
        "datetype": "pdat",
        "mindate": START_YEAR,
        "maxdate": END_YEAR,
        "retstart": start,
        "retmax": batch_size,
        "retmode": "json",
        "email": EMAIL
    }

    if API_KEY:
        search_params["api_key"] = API_KEY

    r = session.get(base_search, params=search_params)
    ids = r.json().get("esearchresult", {}).get("idlist", [])
    pmids.extend(ids)

    time.sleep(0.35)

print(f"\nTotal PMIDs downloaded: {len(pmids)}\n")

##############################################################
# FETCH METADATA
##############################################################

fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
records = []
fetch_batch = 200

for i in tqdm(range(0, len(pmids), fetch_batch), desc="Fetching Metadata"):
    ids_chunk = ",".join(pmids[i:i+fetch_batch])

    fetch_params = {
        "db": "pubmed",
        "id": ids_chunk,
        "retmode": "xml",
        "email": EMAIL
    }

    if API_KEY:
        fetch_params["api_key"] = API_KEY

    response = session.get(fetch_url, params=fetch_params)

    try:
        root = ET.fromstring(response.content)

        for article in root.findall(".//PubmedArticle"):
            try:
                title = article.findtext(".//ArticleTitle")

                abstract_parts = [a.text for a in article.findall(".//AbstractText") if a.text]
                abstract = " ".join(abstract_parts)

                journal = article.findtext(".//Journal/Title")

                year = article.findtext(".//PubDate/Year")
                if not year:
                    year = article.findtext(".//PubDate/MedlineDate")

                pmid = article.findtext(".//PMID")

                doi = ""
                for aid in article.findall(".//ArticleId"):
                    if aid.attrib.get("IdType") == "doi":
                        doi = aid.text
                        break

                authors = []
                for author in article.findall(".//Author"):
                    last = author.findtext("LastName")
                    first = author.findtext("ForeName")
                    if last and first:
                        authors.append(f"{first} {last}")

                author_string = "; ".join(authors)
                url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

                records.append({
                    "PMID": pmid,
                    "DOI": doi,
                    "Title": title,
                    "Authors": author_string,
                    "Journal": journal,
                    "Year": year,
                    "Abstract": abstract,
                    "PubMed URL": url
                })
            except Exception:
                continue

    except ET.ParseError:
        print(f"\nXML parsing error for chunk. Skipping batch.")

    time.sleep(0.35)

##############################################################
# SAVE AND DOWNLOAD
##############################################################

df = pd.DataFrame(records)
output_filename = f"PubMed_Bee_Toxicogenomics_{START_YEAR}_{END_YEAR}.xlsx"
df.to_excel(output_filename, index=False, engine='openpyxl')

print("\nDone Extraction!")
print(f"Total metadata records extracted: {len(df)}")

# Download directly to your local computer from Colab
files.download(output_filename)

Querying NCBI to find total record count...

Total PubMed records found (2010-2026): 420



Fetching PMIDs: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]



Total PMIDs downloaded: 420



Fetching Metadata: 100%|██████████| 3/3 [00:04<00:00,  1.60s/it]


Done Extraction!
Total metadata records extracted: 420


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import requests
import pandas as pd
import re
import time
from tqdm import tqdm
from google.colab import files

##############################################################
# USER SETTINGS
##############################################################

EMAIL = "simon.anguzu@students.mak.ac.ug"

# Category 1: Bee Species / Taxa (Supports space variations like "bumble bee")
BEE_REGEX = re.compile(
    r'(honey\s*bee|honeybee|apis|apis\s+mellifera|apis\s+cerana|apis\s+dorsata|'
    r'stingless\s+bee|meliponini|melipona|meliponula|tetragonisca|bumble\s*bee|bumblebee|bombus|'
    r'bombus\s+terrestris|bombus\s+impatiens|non-apis|wild\s+bee|apoidea)',
    re.IGNORECASE
)

# Category 2: Agrochemicals / Pesticides / Specific Chemicals
PESTICIDE_REGEX = re.compile(
    r'(pesticide|agrochemical|insecticide|neonicotinoid|imidacloprid|thiamethoxam|'
    r'clothianidin|acetamiprid|thiacloprid|flupyradifurone|fungicide|herbicide|'
    r'glyphosate|acaricide|fipronil|organophosphate|pyrethroid|carbamate|'
    r'chlorpyrifos|deltamethrin)',
    re.IGNORECASE
)

# Category 3: Omics / Gene Expression Technologies
OMICS_REGEX = re.compile(
    r'(transcriptom|rna\s*-?\s*seq|rna\s+sequencing|gene\s+expression|differentially\s+expressed|'
    r'deg|degs|microarray|proteom|metabolom|multi-omics|multiomics|ecotoxicogenom|'
    r'toxicogenom|next\s+generation\s+sequencing|ngs)',
    re.IGNORECASE
)

##############################################################
# CROSSREF API FETCH (USING MULTI-QUERY SEARCH)
##############################################################

url = "https://api.crossref.org/works"

headers = {
    "User-Agent": f"ComprehensiveBeeToxicogenomics/3.0 (mailto:{EMAIL})"
}

# Broad query targeting all key areas simultaneously
params = {
    "query": "bee imidacloprid neonicotinoid pesticide gene expression transcriptome",
    "filter": "from-pub-date:2010-01-01,until-pub-date:2026-12-31,type:journal-article",
    "rows": 100,
    "offset": 0
}

print("Searching Crossref repository...")
response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

total_candidates = response.json()["message"]["total-results"]
print(f"Total candidate articles retrieved for screening: {total_candidates}\n")

##############################################################
# FLEXIBLE THREE-WAY CATEGORY FILTERING
##############################################################

filtered_records = []
batch_size = 100
max_candidates = min(total_candidates, 5000)  # Expanded candidate window

for offset in tqdm(range(0, max_candidates, batch_size), desc="Extracting & Validating Papers"):
    params["offset"] = offset
    resp = requests.get(url, headers=headers, params=params)

    if resp.status_code == 200:
        items = resp.json().get("message", {}).get("items", [])

        for item in items:
            title = item.get("title", [""])[0] if item.get("title") else ""
            abstract = item.get("abstract", "")

            # Combine Title + Abstract for strict evaluation
            combined_text = f"{title} {abstract}"

            # Fallback evaluation: Check if matches Bees AND (Pesticides OR Omics) in Title/Abstract
            has_bee = bool(BEE_REGEX.search(combined_text))
            has_pesticide = bool(PESTICIDE_REGEX.search(combined_text))
            has_omics = bool(OMICS_REGEX.search(combined_text))

            # Save if matches all 3 categories OR matches Bee + (Pesticide/Omics) in Title
            if (has_bee and has_pesticide and has_omics) or (has_bee and (has_pesticide or has_omics)):

                pub_year = ""
                if "published" in item and "date-parts" in item["published"]:
                    pub_year = item["published"]["date-parts"][0][0]
                elif "published-print" in item and "date-parts" in item["published-print"]:
                    pub_year = item["published-print"]["date-parts"][0][0]

                journal = item.get("container-title", [""])[0] if item.get("container-title") else ""

                authors = []
                for author in item.get("author", []):
                    given = author.get("given", "")
                    family = author.get("family", "")
                    authors.append(f"{given} {family}".strip())
                author_str = "; ".join(authors)

                filtered_records.append({
                    "DOI": item.get("DOI", ""),
                    "Title": title,
                    "Authors": author_str,
                    "Journal": journal,
                    "Year": pub_year,
                    "Publisher": item.get("publisher", ""),
                    "URL": item.get("URL", "")
                })

    time.sleep(0.2)

##############################################################
# SAVE & EXPORT
##############################################################

df = pd.DataFrame(filtered_records)
df = df.drop_duplicates(subset=["DOI"], keep="first")

filename = "Complete_Bee_Toxicogenomics_Crossref_2010_2026.xlsx"
df.to_excel(filename, index=False, engine='openpyxl')

print(f"\nExtraction Complete!")
print(f"Total papers extracted: {len(df)}")
print(df.head(3))

# Auto-download in Google Colab
files.download(filename)

Searching Crossref repository...
Total candidate articles retrieved for screening: 877507



Extracting & Validating Papers: 100%|██████████| 50/50 [03:06<00:00,  3.73s/it]


Extraction Complete!
Total papers extracted: 401
                              DOI  \
0              10.1242/jeb.127472   
1  10.1016/j.jinsphys.2016.03.004   
2              10.1242/jeb.230060   

                                               Title  \
0  A neonicotinoid, imidacloprid, impairs honey b...   
1  Sperm viability and gene expression in honey b...   
2  Correction: The neonicotinoid imidacloprid imp...   

                                             Authors  \
0                         Erica Zhang; James C. Nieh   
1  Veeranan Chaimanee; Jay D. Evans; Yanping Chen...   
2                         Erica Zhang; James C. Nieh   

                           Journal  Year                  Publisher  \
0  Journal of Experimental Biology  2015  The Company of Biologists   
1     Journal of Insect Physiology  2016                Elsevier BV   
2  Journal of Experimental Biology  2020  The Company of Biologists   

                                              URL  
0             

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import requests
import pandas as pd
import time
from tqdm import tqdm
from google.colab import files

##############################################################
# USER SETTINGS & CREDENTIALS
##############################################################

ELSEVIER_API_KEY = "7f34bdaaf30d51ab3d181537aa644f7c"
INST_TOKEN = ""  # Optional: Add institutional token here if provided by Makerere library

# Comprehensive Scopus search query matching your thesis scope
SCOPUS_QUERY = (
    'TITLE-ABS-KEY(('
    '"honey bee*" OR "honeybee*" OR "Apis" OR "Apis mellifera" OR "Apis cerana" OR "Apis dorsata" OR '
    '"stingless bee*" OR "Meliponini" OR "Melipona" OR "Meliponula" OR "bumblebee*" OR "Bombus" OR '
    '"Bombus terrestris" OR "Bombus impatiens" OR "non-Apis" OR "wild bee*" OR "pollinator*" OR "Apoidea"'
    ') AND ('
    '"pesticide*" OR "agrochemical*" OR "insecticide*" OR "neonicotinoid*" OR "imidacloprid" OR '
    '"thiamethoxam" OR "clothianidin" OR "acetamiprid" OR "thiacloprid" OR "flupyradifurone" OR '
    '"fungicide*" OR "herbicide*" OR "glyphosate" OR "acaricide*" OR "fipronil" OR "organophosphate*" OR '
    '"pyrethroid*" OR "carbamate*" OR "chlorpyrifos" OR "deltamethrin"'
    ') AND ('
    '"transcriptom*" OR "RNA-Seq" OR "RNA sequencing" OR "gene expression" OR "differentially expressed" OR '
    '"DEG" OR "DEGs" OR "microarray*" OR "proteom*" OR "metabolom*" OR "multi-omics" OR "multiomics" OR '
    '"ecotoxicogenom*" OR "toxicogenom*" OR "next generation sequencing" OR "NGS"'
    ')) AND PUBYEAR AFT 2009 AND PUBYEAR BEF 2027'
)

url = "https://api.elsevier.com/content/search/scopus"

headers = {
    "X-ELS-APIKey": ELSEVIER_API_KEY,
    "Accept": "application/json"
}
if INST_TOKEN:
    headers["X-ELS-Insttoken"] = INST_TOKEN

params = {
    "query": SCOPUS_QUERY,
    "count": 25,  # Elsevier Scopus API maximum batch size per call
    "start": 0
}

##############################################################
# CONNECTION & COUNT TEST
##############################################################

print("Connecting to Elsevier Scopus API...")
r = requests.get(url, headers=headers, params=params)

if r.status_code == 401:
    print("\n❌ 401 Unauthorized Error: Key rejected by Elsevier.")
    print("Response:", r.text)
    r.raise_for_status()

r.raise_for_status()
data = r.json()

total_results = int(data["search-results"]["opensearch:totalResults"])
print(f"Connected successfully!")
print(f"Total Scopus records found (2010–2026): {total_results}\n")

##############################################################
# BATCH EXTRACTION
##############################################################

records = []
batch_size = 25
max_records = min(total_results, 5000)  # Fetch up to 5,000 matches

for start in tqdm(range(0, max_records, batch_size), desc="Downloading Scopus Metadata"):
    params["start"] = start
    resp = requests.get(url, headers=headers, params=params)

    if resp.status_code == 200:
        entries = resp.json().get("search-results", {}).get("entry", [])

        for entry in entries:
            # Extract DOI
            doi = entry.get("prism:doi", "")

            # Extract Title
            title = entry.get("dc:title", "")

            # Extract Authors
            authors = entry.get("dc:creator", "")

            # Extract Journal Name
            journal = entry.get("prism:publicationName", "")

            # Extract Publication Year
            cover_date = entry.get("prism:coverDate", "")
            pub_year = cover_date[:4] if cover_date and len(cover_date) >= 4 else ""

            # Extract Abstract (if provided by API payload)
            abstract = entry.get("dc:description", "")

            # Extract Scopus URL / EID
            eid = entry.get("eid", "")
            scopus_link = entry.get("link", [{}, {}])[0].get("@href", "") if entry.get("link") else ""

            records.append({
                "EID": eid,
                "DOI": doi,
                "Title": title,
                "Authors": authors,
                "Journal": journal,
                "Year": pub_year,
                "Abstract": abstract,
                "Scopus_URL": scopus_link
            })

    time.sleep(0.3)  # Respect Elsevier API rate limits

##############################################################
# EXPORT TO EXCEL
##############################################################

df = pd.DataFrame(records)

# Drop any duplicate records by DOI
df = df.drop_duplicates(subset=["DOI"], keep="first")

filename = "Scopus_Bee_Toxicogenomics_2010_2026.xlsx"
df.to_excel(filename, index=False, engine='openpyxl')

print(f"\nExtraction Complete! Downloaded {len(df)} unique Scopus records.")
print(df.head(3))

# Trigger automatic browser download in Colab
files.download(filename)

Connecting to Elsevier Scopus API...
Connected successfully!
Total Scopus records found (2010–2026): 634




Extraction Complete! Downloaded 631 unique Scopus records.
                   EID                         DOI  \
0  2-s2.0-105027735505  10.1186/s12302-025-01280-w   
1  2-s2.0-105035440468  10.1038/s41598-026-42605-w   
2  2-s2.0-105038461319  10.1186/s12864-026-12770-3   

                                               Title       Authors  \
0  Synergistic impacts of flupyradifurone and uni...       Chen L.   
1  Metagenomic and gene expression patterns in de...    Nearman A.   
2  The diversity of insecticide target site mutat...  Verbinnen G.   

                         Journal  Year Abstract  \
0  Environmental Sciences Europe  2026            
1             Scientific Reports  2026            
2                   BMC Genomics  2026            

                                          Scopus_URL  
0  https://api.elsevier.com/content/abstract/scop...  
1  https://api.elsevier.com/content/abstract/scop...  
2  https://api.elsevier.com/content/abstract/scop...  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import pandas as pd
import re
from google.colab import files

##############################################################
# 1. LOAD DATASETS & TRACK SOURCE
##############################################################

print("Loading datasets...")
df_scopus = pd.read_excel("Scopus_Bee_Toxicogenomics_2010_2026.xlsx")
df_pubmed = pd.read_excel("PubMed_Bee_Toxicogenomics_2010_2026.xlsx")
df_crossref = pd.read_excel("Complete_Bee_Toxicogenomics_Crossref_2010_2026.xlsx")

# Add a column to track origin before merging
df_scopus['Source'] = 'Scopus'
df_pubmed['Source'] = 'PubMed'
df_crossref['Source'] = 'Crossref'

print(f"Initial counts:")
print(f" - Scopus: {len(df_scopus)}")
print(f" - PubMed: {len(df_pubmed)}")
print(f" - Crossref: {len(df_crossref)}")

# Combine into a single master DataFrame
df_combined = pd.concat([df_scopus, df_pubmed, df_crossref], ignore_index=True)
total_raw = len(df_combined)
print(f"\nTotal raw records combined: {total_raw}")

##############################################################
# 2. STANDARDIZE IDENTIFIERS FOR ACCURATE MATCHING
##############################################################

# Standardize DOI: lowercase, strip whitespace, remove 'https://doi.org/' prefixes
def clean_doi(doi):
    if pd.isna(doi) or not str(doi).strip():
        return None
    doi_str = str(doi).lower().strip()
    doi_str = re.sub(r'^https?://(dx\.)?doi\.org/', '', doi_str)
    return doi_str if doi_str else None

# Standardize Title: lowercase, remove punctuation and extra spaces
def clean_title(title):
    if pd.isna(title) or not str(title).strip():
        return None
    title_str = str(title).lower().strip()
    title_str = re.sub(r'[^a-z0-9]', '', title_str)  # Keep only alphanumeric chars
    return title_str if title_str else None

df_combined['clean_doi'] = df_combined['DOI'].apply(clean_doi)
df_combined['clean_title'] = df_combined['Title'].apply(clean_title)

##############################################################
# 3. DEDUPLICATION PASSES
##############################################################

# Pass 1: Deduplicate by cleaned DOI (where DOI exists)
df_has_doi = df_combined[df_combined['clean_doi'].notna()].copy()
df_no_doi = df_combined[df_combined['clean_doi'].isna()].copy()

# Keep first occurrence, but consolidate sources
df_dedup_doi = df_has_doi.drop_duplicates(subset=['clean_doi'], keep='first')

# Recombine with records that lacked a DOI
df_pass1 = pd.concat([df_dedup_doi, df_no_doi], ignore_index=True)

# Pass 2: Deduplicate remaining records by normalized Title
df_final = df_pass1.drop_duplicates(subset=['clean_title'], keep='first').copy()

# Drop temporary cleaning columns
df_final = df_final.drop(columns=['clean_doi', 'clean_title'])

duplicates_removed = total_raw - len(df_final)

##############################################################
# 4. REPORT & EXPORT
##############################################################

print("\n" + "="*40)
print("DEDUPLICATION SUMMARY (FOR PRISMA)")
print("="*40)
print(f"Total initial records:     {total_raw}")
print(f"Duplicates identified:     {duplicates_removed}")
print(f"Final unique records:      {len(df_final)}")
print("="*40)

output_filename = "Deduplicated_Bee_Toxicogenomics_Master_2010_2026.xlsx"
df_final.to_excel(output_filename, index=False, engine='openpyxl')

print(f"\nSaved clean dataset as: {output_filename}")
files.download(output_filename)

Loading datasets...
Initial counts:
 - Scopus: 631
 - PubMed: 420
 - Crossref: 401

Total raw records combined: 1452

DEDUPLICATION SUMMARY (FOR PRISMA)
Total initial records:     1452
Duplicates identified:     486
Final unique records:      966

Saved clean dataset as: Deduplicated_Bee_Toxicogenomics_Master_2010_2026.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>